# Lorenz-63 Dynamics Visualization

This notebook visualizes the chaotic Lorenz-63 system and the data generation process:
1. 3D attractor visualization
2. Time series of state variables (X, Y, Z, W_L)
3. Sparse observation sampling
4. Forcing corruption (CS1 vs CS2)
5. Trajectory divergence: true model vs noisy model

In [ ]:
# --- Setup: Colab detection ---
import sys, os
if 'google.colab' in str(get_ipython()):
    !git clone https://github.com/rfablet/4dvarnet-fm-opencode.git
    %cd 4dvarnet-fm-opencode
    !pip install -r requirements.txt
else:
    sys.path.append(os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from data.lorenz63 import Lorenz63Config, Lorenz63Dataset, generate_long_trajectory, generate_corrupted_forcing

device = torch.device('cpu')
print(f'Device: {device}')

In [ ]:
# --- Parameters (edit these) ---
SEED = 42
DURATION = 30.0  # seconds
print(f'Seed: {SEED}, Duration: {DURATION}s')

## 1. Generate Trajectory

In [ ]:
cfg = Lorenz63Config(case=1, num_windows=1, T_max=DURATION, seed=SEED)
dataset = Lorenz63Dataset(cfg)
window = dataset[0]

trajectory_full = window['true_state'].numpy()
forcing_true = window['forcing_true'].numpy()
trajectory_with_forcing = np.concatenate([trajectory_full, forcing_true.reshape(-1, 1)], axis=1)
time_grid = cfg.time_grid
num_steps = len(trajectory_full)
num_obs = window['obs_mask'].sum().item()

print(f'Trajectory: {num_steps} steps over {DURATION:.1f}s')
print(f'Observations: {num_obs} sparse samples (every {cfg.obs_interval} steps)')
print(f'Observation ratio: {100.0 * num_obs / num_steps:.1f}%')

## 2. 3D Attractor

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
X, Y, Z = trajectory_full[:, 0], trajectory_full[:, 1], trajectory_full[:, 2]
time_steps = np.arange(len(X))
scatter = ax.scatter(X, Y, Z, c=time_steps, cmap='viridis', s=1, alpha=0.6)
ax.plot(X, Y, Z, color='gray', alpha=0.2, linewidth=0.5)
ax.set_xlabel('X', fontsize=12)
ax.set_ylabel('Y', fontsize=12)
ax.set_zlabel('Z', fontsize=12)
ax.set_title('Lorenz-63 Attractor (3D Phase Space)', fontsize=14, fontweight='bold')
cbar = plt.colorbar(scatter, ax=ax, pad=0.1, shrink=0.8)
cbar.set_label('Time Step', fontsize=11)
plt.tight_layout()
plt.show()

## 3. Time Series

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 10))
labels = ['X', 'Y', 'Z', r'$W_L$ (Forcing)']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for i, (ax, label, color) in enumerate(zip(axes, labels, colors)):
    ax.plot(time_grid, trajectory_with_forcing[:, i], color=color, linewidth=1.5)
    ax.set_ylabel(label, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_xlim(time_grid[0], time_grid[-1])
axes[-1].set_xlabel('Time (s)', fontsize=12)
axes[0].set_title('Lorenz-63 State Variables (Time Series)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Sparse Observations

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(time_grid, trajectory_full[:, 0], color='black', linewidth=2, label='True State (X)', alpha=0.8)
obs_times = time_grid[window['obs_mask'].numpy()]
obs_values = window['obs'][window['obs_mask'].numpy(), 0].numpy()
ax.scatter(obs_times, obs_values, color='red', s=50, zorder=5,
           label=f'Observations (N={len(obs_times)})', edgecolors='darkred')
ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('X Component', fontsize=12)
ax.set_title('Sparse Observations (every 20 time steps)', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=11, framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

## 5. Forcing Comparison (CS1 vs CS2)

In [ ]:
cfg_cs1 = Lorenz63Config(case=1, num_windows=1, T_max=5.0, seed=123)
cfg_cs2 = Lorenz63Config(case=2, num_windows=1, T_max=5.0, seed=123, param_bias=0.05)
forcing_cs1 = Lorenz63Dataset(cfg_cs1)[0]['forcing_true'].numpy()
forcing_cs2 = Lorenz63Dataset(cfg_cs2)[0]['forcing_corrupted'].numpy()
time_grid_cs = cfg_cs1.time_grid

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
axes[0].plot(time_grid_cs, forcing_cs1, color='green', linewidth=2, label='CS1: True Forcing', alpha=0.8)
axes[0].plot(time_grid_cs, forcing_cs2, color='orange', linewidth=2, linestyle='--', label='CS2: Corrupted Forcing', alpha=0.8)
axes[0].set_ylabel(r'$W_L$ (Forcing)', fontsize=12)
axes[0].set_title('Case Study Comparison: Forcing Corruption', fontsize=14, fontweight='bold')
axes[0].legend(loc='upper right', fontsize=11, framealpha=0.9)
axes[0].grid(True, alpha=0.3, linestyle='--')
difference = forcing_cs2 - forcing_cs1
axes[1].plot(time_grid_cs, difference, color='purple', linewidth=1.5)
axes[1].fill_between(time_grid_cs, 0, difference, color='purple', alpha=0.3, label='Corruption Magnitude')
axes[1].axhline(0, color='black', linestyle=':', linewidth=1)
axes[1].set_xlabel('Time (s)', fontsize=12)
axes[1].set_ylabel(r'$\Delta W_L$ (Difference)', fontsize=12)
axes[1].legend(loc='upper right', fontsize=11, framealpha=0.9)
axes[1].grid(True, alpha=0.3, linestyle='--')
std_diff = np.std(difference)
axes[1].text(0.02, 0.98, f'Corruption Std: {std_diff:.4f}',
             transform=axes[1].transAxes, fontsize=10,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
plt.tight_layout()
plt.show()

## 6. Trajectory Divergence: True Model vs Noisy Model

Forward-simulate the Lorenz-63 from the same initial condition with two different model configurations to see how they diverge:
- **True model**: Correct parameters (σ=10, ρ=28, β=8/3) and true forcing
- **Noisy model**: Biased parameters (5%) and corrupted forcing (OU process)

This divergence is the root cause of poor DA performance in CS2.

In [ ]:
div_seed = 123
div_duration = 10.0
div_cfg = Lorenz63Config(case=1, num_windows=1, T_max=div_duration, seed=div_seed)
n_steps = div_cfg.num_steps
dt = div_cfg.dt

sigma_b, rho_b, beta_b = Lorenz63Config(case=2, param_bias=0.05).biased_params
print(f'True parameters:  sigma={div_cfg.sigma_true}, rho={div_cfg.rho_true}, beta={div_cfg.beta_true:.4f}')
print(f'Biased parameters: sigma={sigma_b:.2f}, rho={rho_b:.2f}, beta={beta_b:.4f}')

traj_true = generate_long_trajectory(
    n_steps, dt, div_seed,
    div_cfg.sigma_true, div_cfg.rho_true, div_cfg.beta_true,
    div_cfg.gamma, div_cfg.W_L_bar, div_cfg.c1, div_cfg.c2,
    div_cfg.sigma_0, div_cfg.sigma_L, device,
)

traj_biased = generate_long_trajectory(
    n_steps, dt, div_seed,
    sigma_b, rho_b, beta_b,
    div_cfg.gamma, div_cfg.W_L_bar, div_cfg.c1, div_cfg.c2,
    div_cfg.sigma_0, div_cfg.sigma_L, device,
)

W_L_true = traj_true[:, 3]
W_L_corrupted = generate_corrupted_forcing(W_L_true, n_steps, dt, div_cfg.tau_eta, div_cfg.sigma_eta, div_seed + 2, device)
traj_true_state = traj_true[:, :3].numpy()
traj_biased_state = traj_biased[:, :3].numpy()
div_time = div_cfg.time_grid

print(f'\nFinal RMSE between trajectories: {np.sqrt(np.mean((traj_true_state - traj_biased_state)**2)):.4f}')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
components = ['X', 'Y', 'Z']
for i, (ax, comp) in enumerate(zip(axes, components)):
    ax.plot(div_time, traj_true_state[:, i], color='blue', linewidth=1.5, label='True model', alpha=0.8)
    ax.plot(div_time, traj_biased_state[:, i], color='red', linewidth=1.5, linestyle='--', label='Noisy model', alpha=0.8)
    ax.fill_between(div_time, traj_true_state[:, i], traj_biased_state[:, i], alpha=0.15, color='purple')
    ax.set_ylabel(comp, fontsize=12, fontweight='bold')
    ax.legend(loc='upper right', fontsize=11)
    ax.grid(True, alpha=0.3, linestyle='--')
axes[0].set_title('Trajectory Divergence: True Model vs Noisy Model (5% bias + OU forcing)',
                  fontsize=14, fontweight='bold')
axes[-1].set_xlabel('Time (s)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={'projection': '3d'})
ax.plot(traj_true_state[:, 0], traj_true_state[:, 1], traj_true_state[:, 2],
        color='blue', linewidth=1, label='True model', alpha=0.6)
ax.plot(traj_biased_state[:, 0], traj_biased_state[:, 1], traj_biased_state[:, 2],
        color='red', linewidth=1, linestyle='--', label='Noisy model', alpha=0.6)
ax.set_xlabel('X', fontsize=12)
ax.set_ylabel('Y', fontsize=12)
ax.set_zlabel('Z', fontsize=12)
ax.set_title('3D Attractor: True vs Noisy Model', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(div_time, W_L_true.numpy(), color='green', linewidth=1.5, label='True Forcing W_L', alpha=0.8)
ax.plot(div_time, W_L_corrupted.numpy(), color='orange', linewidth=1.5, linestyle='--', label='Corrupted W_L*', alpha=0.8)
ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel(r'$W_L$', fontsize=12)
ax.set_title('Forcing: True vs Corrupted (OU process)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

### Summary
- The Lorenz-63 system is chaotic: small parameter biases cause large trajectory divergence
- Sparse observations (every 20 steps, ~5% of states) make reconstruction challenging
- CS2 adds model mismatch (biased parameters + corrupted forcing), which classical DA cannot correct
- This motivates data-driven approaches like 4DVarNet-FM